## Content Filters

**Adding the Boiler Plate Code**

In [ ]:
import os
import boto3
import botocore
import json

aws_region = "us-east-1"
guardrail_id = os.environ.get("BEDROCK_GUARDRAIL_ID")
guardrail_version = "DRAFT"

# Created the Bedrock client
bedrock_client = boto3.client(
    service_name="bedrock",
    region_name=aws_region,
)

# Created the Bedrock runtime client
bedrock_runtime_client = boto3.client(
    service_name="bedrock-runtime",
    region_name=aws_region
    )

# Print the URL for the Bedrock client
print(bedrock_client.meta.endpoint_url)
print(bedrock_runtime_client.meta.endpoint_url)


**Delete the GuardRails**

In [ ]:
# Set guardrail_id after running create_guardrail
if guardrail_id:
    response = bedrock_client.delete_guardrail(
        guardrailIdentifier=guardrail_id,
    )
    print(response)
else:
    print("Set guardrail_id before deleting a guardrail.")


In [ ]:
message = "I need make masala dosa"

result = bedrock_runtime_client.converse(
    # Configuring the Model
    modelId="amazon.nova-pro-v1:0",
    
    # Configuring the Messages
    messages=[
        {
            "role": "user",
            "content": [{"text": message}]
        }
    ],
    
    # Configuring the GuardRails
    guardrailConfig={
        "guardrailIdentifier": bedrock_guardrail["guardrailId"],
        "guardrailVersion": "DRAFT",
        "trace": "enabled"
    },
    
    # Configuring the Inference
    inferenceConfig={
        "maxTokens": 150,
        "temperature": 0.5,
        "topP": 1.0,
    }
)
print("=== Full Response ===")
print(result)

print("\n=== Model Response ===")

response_text = result["output"]["message"]["content"][0]["text"]

print(response_text)


### Implementation

In [ ]:
GUARDRAIL_NAME = "ContentFilter"

ALL_FILTERS_HIGH = [
    {"type": "HATE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
    {"type": "INSULTS", "inputStrength": "HIGH", "outputStrength": "HIGH"},
    {"type": "SEXUAL", "inputStrength": "HIGH", "outputStrength": "HIGH"},
    {"type": "VIOLENCE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
    {"type": "MISCONDUCT", "inputStrength": "HIGH", "outputStrength": "HIGH"},
    {"type": "PROMPT_ATTACK", "inputStrength": "HIGH", "outputStrength": "NONE"},
]


def create_guardrail(guardrail_name, filters):

    response = bedrock_client.create_guardrail(
        name=guardrail_name,

        description=f"Guardrail to filter {guardrail_name} content",

        blockedInputMessaging=(
            "This input message is blocked because "
            "it contains content that is not allowed."
        ),

        blockedOutputsMessaging=(
            "This output message is blocked because "
            "it contains content that is not allowed."
        ),

        contentPolicyConfig={
            "filtersConfig": filters
        }
    )

    guardrail_id = response["guardrailId"]
    guardrail_version = response["version"]

    return guardrail_id, guardrail_version



In [ ]:
#Create the GuardRail
guardrail_id, guardrail_version = create_guardrail("demoAnalyser", ALL_FILTERS_HIGH)
print((guardrail_id, guardrail_version))

In [ ]:
def check_with_guardrail(
    text: str,
    guardrail_id: str,
    guardrail_version: str = "DRAFT"
):
    """
    Evaluate a single text using Amazon Bedrock Guardrails.

    Args:
        text: Text to evaluate.
        guardrail_id: Bedrock Guardrail identifier.
        guardrail_version: Guardrail version. Defaults to DRAFT.

    Returns:
        The complete Guardrail response.
    """

    result = bedrock_runtime_client.apply_guardrail(
        guardrailIdentifier=guardrail_id,
        guardrailVersion=guardrail_version,
        source="INPUT",
        content=[
            {
                "text": {
                    "text": text
                }
            }
        ]
    )

    print("\n" + "=" * 60)
    print("           AMAZON BEDROCK GUARDRAIL RESULT")
    print("=" * 60)

    print(f"Input Text : {text}")
    print(f"Action     : {result.get('action')}")

    if result.get("outputs"):
        print("\nOutput:")
        for output in result["outputs"]:
            print(f"  {output.get('text')}")

    if result.get("assessments"):
        print("\nAssessments:")
        print(json.dumps(result["assessments"], indent=4))

    print("=" * 60)

    return result

In [ ]:
check_with_guardrail("Why all terrorist are muslims , what is the reason behind it", guardrail_id, guardrail_version)


## Denied Topics , Word Filters and PII Redaction

Topic denial


In [ ]:
response = bedrock_client.create_guardrail(
    name="LegalTopicGuardrail",

    description="Guardrail to deny legal-related topics",

    blockedInputMessaging=(
        "This input message is blocked because "
        "legal-related topics are not allowed."
    ),

    blockedOutputsMessaging=(
        "This output message is blocked because "
        "legal-related topics are not allowed."
    ),

    topicPolicyConfig={
        "topicsConfig": [
            {
                "name": "LegalTopic",

                "definition": (
                    "Requests for legal advice, legal interpretation, "
                    "legal representation, or guidance about laws, "
                    "regulations, lawsuits, contracts, or legal rights."
                ),

                "examples": [
                    "Can you give me legal advice about my situation?",
                    "What should I do if I am being sued?",
                    "Can you explain my legal rights?",
                    "Is this contract legally valid?",
                    "What should I do if I receive a court notice?"
                ],

                "type": "DENY"
            }
        ]
    }
)

guardrail_id = response["guardrailId"]
guardrail_version = response["version"]

print("Guardrail ID:", guardrail_id)
print("Guardrail Version:", guardrail_version)

**Application of the Topic GuardRails**

In [ ]:
check_with_guardrail(guardrail_id=guardrail_id, text="i want a legal advicer for the robbery case")


**Blocking the Word Sequence**

In [ ]:
bedrock_guardrail = bedrock_client.create_guardrail(
    name="word_and_regex_guardrail",
    description="Blocks custom words and mobile numbers",
    blockedInputMessaging="This input is blocked",
    blockedOutputsMessaging="This output is blocked",

    # Custom word blocking
    wordPolicyConfig={
        "wordsConfig": [
            {
                "text": "mia khalifa",
                "inputAction": "BLOCK",
                "outputAction": "BLOCK",
                "inputEnabled": True,
                "outputEnabled": True,
            },
            # optional: add common variants
            {
                "text": "miakhalifa",
                "inputAction": "BLOCK",
                "outputAction": "BLOCK",
                "inputEnabled": True,
                "outputEnabled": True,
            },
        ]
    },

    # Regex / sensitive info blocking
    sensitiveInformationPolicyConfig={
        # Built-in phone detector (recommended)
        "piiEntitiesConfig": [
            {
                "type": "PHONE",
                "action": "BLOCK",
                "inputAction": "BLOCK",
                "outputAction": "BLOCK",
                "inputEnabled": True,
                "outputEnabled": True,
            }
        ],

        # Custom regex for mobile numbers
        "regexesConfig": [
            {
                "name": "indian-mobile-number",
                "description": "Block Indian 10-digit mobile numbers",
                # Matches: 9876543210, +91-9876543210, 91 9876543210
                "pattern": r"(?:\+91|91)?[\s-]?[6-9]\d{9}",
                "action": "BLOCK",
                "inputAction": "BLOCK",
                "outputAction": "BLOCK",
                "inputEnabled": True,
                "outputEnabled": True,
            },
            {
                "name": "generic-mobile-number",
                "description": "Block generic phone-like numbers",
                "pattern": r"\b\+?[0-9][0-9\s-]{7,14}[0-9]\b",
                "action": "BLOCK",
                "inputAction": "BLOCK",
                "outputAction": "BLOCK",
                "inputEnabled": True,
                "outputEnabled": True,
            },
        ],
    },
)

print(bedrock_guardrail)

In [ ]:
word_test = bedrock_runtime_client.apply_guardrail(
    guardrailIdentifier=bedrock_guardrail["guardrailId"],
    guardrailVersion=bedrock_guardrail["version"],
    source="INPUT",
    content=[
        {"text": {"text": "Tell me about Jayden James with number 9876543210"}}
    ],
    outputScope="FULL",
)

print(word_test["action"])
print(word_test.get("outputs"))
print(word_test.get("assessments"))
